# Disease Spread Network Analysis: Healthcare Data Cleaning & Graph ETL

**Course / Project Title:** Disease Spread Network Analysis  
**Domain:** Healthcare Epidemiology & Graph Analytics  
**Dataset:** Kaggle Healthcare Dataset (55,500 Real Patient Records)  
**Source URL:** [https://www.kaggle.com/datasets/prasad22/healthcare-dataset](https://www.kaggle.com/datasets/prasad22/healthcare-dataset)  

> *"A Disease Spread Network Analysis is a healthcare data analytics system that studies the relationships between diseases, locations, and healthcare facilities to understand how diseases are distributed and connected across regions. It uses graph databases, GIS visualization, and data processing techniques to identify disease clusters, analyze geographic patterns, and provide insights into disease occurrence and spread trends."*

---
### Pipeline Overview:
1. **Raw Dataset Ingestion**: Load the 55,500-record raw healthcare dataset.
2. **Data Quality Profiling**: Profile raw noise (534 duplicate records, 106 negative billing anomalies, mixed text casing, unparsed date strings).
3. **Data Cleaning Pipeline**: Remove duplicates, sanitize casing, clean hospital delimiters, filter billing anomalies, and calculate clinical length-of-stay.
4. **Healthcare Network Engineering**: Aggregate facility caseloads by medical condition and compute hospital capacity metrics.
5. **Neo4j Graph Database Export**: Export structured entity CSVs for Nodes and Relationships.
6. **Cypher Query Execution**: Query the healthcare network to discover disease prevalence hubs and facility clusters.

## 1. Raw Dataset Ingestion

We load `healthcare_dataset.csv`. If this notebook is executed in a new Google Colab cloud environment where the file has not yet been uploaded, it automatically fetches the dataset from our repository to guarantee seamless execution.

In [ ]:
import os
import urllib.request
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 15)

csv_filename = 'healthcare_dataset.csv'

# Automatic download fallback for Google Colab runtimes
if not os.path.exists(csv_filename):
    print(f"'{csv_filename}' not found locally. Fetching raw Kaggle dataset from repository...")
    url = 'https://raw.githubusercontent.com/shivab-lgtm/disease-spread-network-analysis/main/healthcare_dataset.csv'
    urllib.request.urlretrieve(url, csv_filename)
    print('Download complete!')

raw_df = pd.read_csv(csv_filename)
print(f"Successfully loaded Raw Dataset: {raw_df.shape[0]:,} rows and {raw_df.shape[1]} columns.")
display(raw_df.head(10))


## 2. Profiling the "Dirty Data" (Identifying Quality Issues)

Before performing cleaning transformations, we audit the dataset to document the real-world defects present in the raw data.

In [ ]:
print("=== 1. DUPLICATE PATIENT RECORDS ===")
num_dupes = raw_df.duplicated().sum()
print(f"Total exact duplicate rows: {num_dupes:,}")
display(raw_df[raw_df.duplicated(keep=False)].head(6))

print("\n=== 2. NUMERICAL ANOMALIES (NEGATIVE BILLING AMOUNTS) ===")
neg_billing = raw_df[raw_df['Billing Amount'] < 0]
print(f"Records with negative billing values (< $0): {len(neg_billing):,}")
print(f"Minimum billing amount in raw data: ${raw_df['Billing Amount'].min():,.2f}")
display(neg_billing[['Name', 'Hospital', 'Medical Condition', 'Billing Amount']].head(6))

print("\n=== 3. TEXT CASING DEFECTS IN PATIENT NAMES ===")
sample_casing = raw_df['Name'].head(8).tolist()
for name in sample_casing:
    print(f"  Raw casing: '{name}'")

print("\n=== 4. PUNCTUATION & DELIMITER NOISE IN HOSPITAL NAMES ===")
noisy_hospitals = raw_df[raw_df['Hospital'].str.endswith((',', ';', '.'))]['Hospital'].unique()
print(f"Sample hospital names with trailing delimiters: {list(noisy_hospitals[:5])}")

print("\n=== 5. MEDICAL CONDITION DISTRIBUTION ===")
display(raw_df['Medical Condition'].value_counts().to_frame('Patient Count'))


## 3. Data Cleaning Pipeline

We now execute our automated data cleaning and engineering pipeline:
1. **Deduplication**: Remove all 534 redundant duplicate patient entries.
2. **Text Normalization**: Convert ransom-style erratic name casing into standard title case (`Bobby JacksOn` -> `Bobby Jackson`) and strip whitespace.
3. **Hospital Name Sanitization**: Remove trailing commas, periods, and unstandardized delimiters.
4. **Outlier Filtering & Precision Normalization**: Filter out negative billing anomalies and round financial amounts to standard currency precision (2 decimals).
5. **Temporal Transformation & Stay Duration**: Parse `Date of Admission` and `Discharge Date` into ISO-8601 datetimes and compute `Length_of_Stay_Days`.

In [ ]:
clean_df = raw_df.copy()

# Step 1: Remove Duplicate Rows
clean_df = clean_df.drop_duplicates().copy()

# Step 2: Normalize Text Fields (Title Case & Whitespace Trimming)
clean_df['Name'] = clean_df['Name'].str.title().str.strip()
clean_df['Doctor'] = clean_df['Doctor'].str.title().str.strip()
clean_df['Hospital'] = clean_df['Hospital'].str.rstrip(' ,.;').str.strip()

# Step 3: Numerical Validation & Currency Rounding
clean_df = clean_df[clean_df['Billing Amount'] >= 0].copy()
clean_df['Billing Amount'] = clean_df['Billing Amount'].round(2)

# Step 4: Temporal Standardization & Length of Stay Feature Engineering
clean_df['Date of Admission'] = pd.to_datetime(clean_df['Date of Admission'], format='%Y-%m-%d', errors='coerce')
clean_df['Discharge Date'] = pd.to_datetime(clean_df['Discharge Date'], format='%Y-%m-%d', errors='coerce')
clean_df['Length_of_Stay_Days'] = (clean_df['Discharge Date'] - clean_df['Date of Admission']).dt.days

# Step 5: Clinical Stay Cohort Classification
def classify_stay(days):
    if days <= 3:
        return 'Short Stay (1-3d)'
    elif days <= 10:
        return 'Moderate Stay (4-10d)'
    else:
        return 'Extended Stay (>10d)'

clean_df['Stay_Category'] = clean_df['Length_of_Stay_Days'].apply(classify_stay)

print(f"Data Cleaning Complete! Validated clean records: {len(clean_df):,} (Filtered from {len(raw_df):,} raw records)")
display(clean_df[['Name', 'Age', 'Gender', 'Medical Condition', 'Hospital', 'Date of Admission', 'Discharge Date', 'Length_of_Stay_Days', 'Billing Amount']].head(10))


## 4. Healthcare Network Engineering (Disease & Facility Aggregation)

To analyze how diseases are distributed across healthcare facilities, we aggregate patient records into a **Bipartite Healthcare Network** linking Hospitals to Medical Conditions.

In [ ]:
# Aggregate Hospital-to-Condition Network Relationships
hospital_network = clean_df.groupby(['Hospital', 'Medical Condition']).agg(
    patient_count=('Name', 'count'),
    avg_billing=('Billing Amount', 'mean'),
    avg_stay_days=('Length_of_Stay_Days', 'mean')
).reset_index()

hospital_network['avg_billing'] = hospital_network['avg_billing'].round(2)
hospital_network['avg_stay_days'] = hospital_network['avg_stay_days'].round(1)

# Facility Inpatient Caseload Volume Classification
hospital_totals = clean_df.groupby('Hospital')['Name'].count().to_frame('total_inpatient_cases').reset_index()

def classify_facility_load(cases):
    if cases >= 5:
        return 'High Volume Regional Hub'
    elif cases >= 2:
        return 'Moderate Community Facility'
    else:
        return 'Peripheral Clinic'

hospital_totals['facility_tier'] = hospital_totals['total_inpatient_cases'].apply(classify_facility_load)

print(f"Derived Healthcare Network Relationships: {len(hospital_network):,} unique hospital-condition edges.")
display(hospital_network.head(10))
display(hospital_totals['facility_tier'].value_counts().to_frame('Facility Count by Tier'))


## 5. Export Structured Entities for Neo4j Graph Database

We now export the clean dataset and the relational graph tables (Nodes and Relationships) formatted for Neo4j ingestion.

In [ ]:
# 1. Clean Master Dataset
clean_df.to_csv('cleaned_healthcare_dataset.csv', index=False)

# 2. Node Table: Hospital Nodes
nodes_hospital = hospital_totals[['Hospital', 'total_inpatient_cases', 'facility_tier']].copy()
nodes_hospital.columns = ['name', 'total_cases', 'tier']
nodes_hospital.to_csv('neo4j_nodes_hospital.csv', index=False)

# 3. Node Table: Medical Condition Nodes
nodes_condition = clean_df[['Medical Condition']].drop_duplicates().copy()
nodes_condition.columns = ['name']
nodes_condition['category'] = 'Chronic / Non-communicable'
nodes_condition.to_csv('neo4j_nodes_condition.csv', index=False)

# 4. Relationship Table: Hospital -> TREATS -> Medical Condition
rel_treats = hospital_network.copy()
rel_treats.columns = ['hospital_name', 'condition_name', 'patient_count', 'avg_billing', 'avg_stay_days']
rel_treats.to_csv('neo4j_rel_hospital_treats_condition.csv', index=False)

print("Successfully exported files for Neo4j & Analysis:")
print(" - cleaned_healthcare_dataset.csv")
print(" - neo4j_nodes_hospital.csv")
print(" - neo4j_nodes_condition.csv")
print(" - neo4j_rel_hospital_treats_condition.csv")


## 6. Neo4j Graph Database Integration & Cypher Queries

Below are the Cypher queries executed in Neo4j Browser to model and query the disease and hospital network:

### Step 6.1: Define Constraints & Indexes
```cypher
CREATE CONSTRAINT IF NOT EXISTS FOR (h:Hospital) REQUIRE h.name IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (c:MedicalCondition) REQUIRE c.name IS UNIQUE;
```

### Step 6.2: Query the Healthcare Network
```cypher
// 1. View hospitals treating Asthma patients with high caseload
MATCH (h:Hospital)-[t:TREATS]->(c:MedicalCondition {name: 'Asthma'})
WHERE t.patient_count >= 2
RETURN h.name, c.name, t.patient_count, t.avg_billing
ORDER BY t.patient_count DESC
LIMIT 10;

// 2. Discover high-volume regional healthcare facilities
MATCH (h:Hospital {tier: 'High Volume Regional Hub'})-[t:TREATS]->(c:MedicalCondition)
RETURN h, t, c
LIMIT 25;
```